# Demo — Condenser Vacuum Loss: PM Compliance & Recurrence Analysis
**Unit 2 · PWR · Event EVT-U2-2024-0847 · 14 July 2024 03:22 UTC**

This notebook demonstrates how the RCA system surfaces **maintenance compliance gaps** and **recurrence patterns** for a multi-hypothesis failure scenario. Target audience: managers and system engineers.

---

### What happened
Unit 2 experienced an **automatic turbine load runback** from 97% to 85% rated power when condenser backpressure reached the 3.0 inHg setpoint. The runback followed a **14-day monotonic backpressure rise** from 1.8 inHg baseline. Hotwell dissolved oxygen was found at **142 ppb** at the time of the event (normal: < 10 ppb).

Three competing hypotheses exist:
| # | Hypothesis | Key evidence |
|---|---|---|
| 1 | **Air in-leakage** through expansion joint | Elevated dissolved oxygen (DO) — diagnostic for air in-leakage only |
| 2 | Condenser tube fouling | Contradicted by recent tube inspection (zero tubes plugged) |
| 3 | Elevated CW inlet temperature | Seasonal rise insufficient to explain 1.2 inHg increase alone |

**True root cause:** Air in-leakage through the turbine exhaust duct expansion joint.  
**Contributing factor:** HVAC fan motor bearing failure (Day -10) elevated condenser pit temperature, accelerating seal degradation.

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

FIXTURE_DIR = Path('test_case_3/fixtures')
RCA_ROOT    = Path('..').resolve()
if str(RCA_ROOT) not in sys.path:
    sys.path.insert(0, str(RCA_ROOT))

event        = json.loads((FIXTURE_DIR / 'event.json').read_text())
pm           = json.loads((FIXTURE_DIR / 'pm_compliance.json').read_text())
kg           = json.loads((FIXTURE_DIR / 'kg_context.json').read_text())
telemetry    = json.loads((FIXTURE_DIR / 'telemetry_summary.json').read_text())
op_ctx       = json.loads((FIXTURE_DIR / 'operational_context.json').read_text())

print('Fixtures loaded.')

---
## Section 1 — Event Summary

In [ ]:
sig = event.get('symptom_signature', {})
params = sig.get('affected_parameters', [])

rows = []
for p in params:
    nr = p.get('normal_range', {})
    rows.append({
        'Parameter':     p['parameter'].replace('_', ' ').title(),
        'Sensor':        p['sensor_id'],
        'Observed':      f"{p['observed_value']} {p.get('unit','')}",
        'Normal range':  f"{nr.get('min','?')} – {nr.get('max','?')} {p.get('unit','')}",
    })

df = pd.DataFrame(rows)
print(f"Event:    {event['event_id']}")
print(f"Asset:    {event['asset_id']}")
print(f"Time:     {event['timestamp_start']}")
print(f"Severity: {event['severity']}")
print()
df.style.set_caption('Affected parameters at time of event')\
        .set_properties(**{'text-align': 'left'})\
        .hide(axis='index')

---
## Section 2 — Preventive Maintenance Compliance

The PM compliance assessment covers the **104-day look-back window** prior to the event.  
Failed items are those directly relevant to the causal chain under investigation.

In [ ]:
checks = pm.get('checks', [])
rows = []
for c in checks:
    rows.append({
        'Check ID':        c['check_id'],
        'Type':            c['check_type'].replace('_', ' ').title(),
        'Status':          '❌ FAIL' if c['status'] == 'fail' else '✅ PASS',
        'Overdue (days)':  int(c['overdue_by_days']) if c['overdue_by_days'] > 0 else '—',
        'Component':       c.get('component_id', '—'),
        'Details':         c['details'][:90] + '…',
    })

df_pm = pd.DataFrame(rows)
s = pm['summary']
print(f"Compliance rate:          {s['compliance_rate']*100:.0f}%  ({s['passed']} passed, {s['failed']} failed)")
print(f"Overall compliance:       {s['overall_compliance'].upper()}")
print(f"Maintenance-induced risk: {s['maintenance_induced_risk'].upper()}")
print(f"Data quality confidence:  {s['data_quality_confidence'].upper()}")
print()
df_pm[['Check ID','Type','Status','Overdue (days)','Details']]\
    .style.set_caption('PM check results — 104-day look-back')\
          .hide(axis='index')

In [ ]:
print('KEY INSIGHT')
print('=' * 60)
print(s.get('notes', ''))

---
## Section 3 — Failure Mode Recurrence Analysis

The TSKR (Temporal Scorer for Knowledge-based Recurrence) module analyses **past corrective action records** to determine whether each failure mode has occurred before, how frequently, and whether the rate is accelerating.

> **Why this matters:** A failure mode with an increasing recurrence rate is a signal that corrective actions from prior events may have been inadequate or that underlying conditions are worsening.

In [ ]:
from orchestrators.tskr_temporal_scorer import TSKRTemporalScorerV1

scorer  = TSKRTemporalScorerV1()
result  = scorer.score(
    event               = event,
    telemetry_summary   = telemetry,
    kg_context          = kg,
    operational_context = op_ctx,
    run_context         = {'run_id': 'DEMO-TC3'},
    pm_compliance       = pm,
)

patterns = result['patterns']
print(f"Scored {len(patterns)} failure mode(s).")

In [ ]:
TREND_LABEL = {
    'increasing':       '⚠️  Increasing (accelerating)',
    'decreasing':       '✅ Decreasing (improving)',
    'stable':           '➡️  Stable',
    'insufficient_data':'—  Insufficient history',
}

fm_labels = {fm['fm_id']: fm['name'] for fm in kg.get('failure_modes', [])}

rows = []
for p in patterns:
    fm_id = p['target_id']
    rows.append({
        'Failure Mode':          fm_labels.get(fm_id, fm_id),
        'Prior Events':          p['recurrence_count'],
        'Trend':                 TREND_LABEL.get(p['recurrence_trend'], p['recurrence_trend']),
        'Unresolved':            p['unresolved_recurrence_count'],
        'Most Recent (days ago)':p.get('most_recent_days_ago') or '—',
        'Contributing CRs':      ', '.join(p.get('contributing_event_ids', [])) or '—',
    })

df_rec = pd.DataFrame(rows)
df_rec.style.set_caption('Recurrence history by failure mode')\
            .hide(axis='index')

> **Recurrence trap:** The most recent similar event (18 months ago) was confirmed as **tube fouling**. However, the two *older* events (36 and 60 months ago) were both confirmed **air in-leakage**. Weighted recurrence analysis correctly favours air in-leakage over tube fouling despite the recency of the fouling event.

---
## Section 4 — Signal Analysis & PM Maintenance Boost

Each failure mode is scored against the telemetry signals observed during the event window.  
When overdue maintenance items share the same physical component as a failure mode, the system applies a **PM boost** to the history score — reflecting that deferred maintenance increases the credibility of that failure mode as a cause.

In [ ]:
rows = []
for p in patterns:
    fm_id = p['target_id']
    flags = p.get('attention_flags', [])
    flag_str = ' | '.join(f'⚠️ {f.replace("_", " ").title()}' for f in flags) if flags else '—'
    rows.append({
        'Failure Mode':      fm_labels.get(fm_id, fm_id),
        'Confidence':        f"{p['confidence']:.2f}",
        'Temporal Relation': p.get('relation', '—'),
        'Matching Signals':  ', '.join(p.get('matching_signal_ids', [])) or '—',
        'Signal Novel':      '✅ New pattern' if p.get('signal_novel') else 'Known pattern',
        'PM Overdue Boost':  f"+{p.get('pm_overdue_boost', 0.0):.2f}" if p.get('pm_overdue_boost', 0) > 0 else '—',
        'Attention Flags':   flag_str,
    })

df_sig = pd.DataFrame(rows)
df_sig.style.set_caption('Pattern scoring results by failure mode')\
            .hide(axis='index')

---
## Section 5 — Telemetry Signal Inventory

The signals below show which sensors had anomalies and which did not. The **absence** of anomalies on tube-side sensors is as diagnostically important as the presence of anomalies on the DO sensor.

In [ ]:
rows = []
for sig in telemetry.get('signals', []):
    anoms = sig.get('anomalies', [])
    if anoms:
        a0      = anoms[0]
        status  = f"⚠️  {len(anoms)} anomaly(ies) — {a0.get('severity','?')} severity"
        pattern = a0.get('pattern', a0.get('tone', '—'))
    else:
        status  = '✅ Within normal limits'
        pattern = '—'
    rows.append({
        'Sensor':      sig['sensor_id'],
        'Parameter':   sig.get('parameter','').replace('_',' ').title(),
        'Status':      status,
        'Pattern':     pattern,
    })

df_tel = pd.DataFrame(rows)
df_tel.style.set_caption('Telemetry signal status during event window')\
            .hide(axis='index')

---
## Section 6 — Analyst Attention Flags

The system automatically generates flags that require analyst review before the RCA can be closed.

In [ ]:
summary = result.get('summary', {})

print('TSKR SUMMARY')
print('─' * 60)
print(f"  Failure modes scored:    {summary.get('n_patterns', 0)}")
print(f"  With temporal support:   {summary.get('n_supported_patterns', 0)}")
print(f"  Novel patterns:          {summary.get('n_novel_patterns', 0)}")
print(f"  Total CR records found:  {summary.get('total_cr_count', 0)}")
print(f"  Unmatched CR rate:       {summary.get('unmatched_cr_rate', 0)*100:.0f}%")
print()

accel_flags = [
    p['target_id'] for p in patterns
    if 'accelerating_recurrence' in p.get('attention_flags', [])
]
novel_flags = [
    p['target_id'] for p in patterns
    if p.get('novel_pattern')
]

if accel_flags:
    print(f"⚠️  ACCELERATING RECURRENCE detected for: {', '.join(accel_flags)}")
    print('   Inter-event intervals are shrinking. Consider escalating PM frequency.')
else:
    print('✅ No accelerating recurrence patterns detected.')

if novel_flags:
    print(f"\n⚠️  NOVEL PATTERNS (no prior history): {', '.join(novel_flags)}")
else:
    print('✅ All failure modes have some historical precedent.')

---
## Section 7 — Analyst Summary

### What the system found

**PM Compliance (104-day window):**  
3 of 4 preventive maintenance checks failed. The failed items — expansion joint inspection (104 days overdue), air ejector surveillance (12 days overdue), and HVAC fan motor PM (60 days overdue) — are all directly relevant to the air in-leakage causal chain. The one passed check (condenser tube inspection, completed 21 days prior with zero tubes plugged) actively *contradicts* the tube fouling hypothesis.

**Recurrence analysis:**  
Air in-leakage has two prior confirmed events (36 and 60 months ago). Tube fouling has one prior confirmed event (18 months ago — the most recent analog). Despite the recency of the fouling event, air in-leakage has the stronger historical pattern for this unit.

**Signal discriminator:**  
The hotwell dissolved oxygen sensor (U2-AIT-0341) at 142 ppb is the key discriminating signal. Elevated DO in the hotwell is **diagnostic for air in-leakage** and is inconsistent with tube fouling, tube leakage, or circulating water temperature effects.

### Recommended actions for analyst review

1. **Confirm helium leak test results** on the expansion joint (WO-2024-12001 was post-event — requires analyst acceptance as confirmatory evidence).
2. **Assess whether the expansion joint inspection deferral** constitutes a programmatic issue requiring a separate corrective action against the PM program.
3. **Evaluate HVAC fan bearing failure** as an independently correctable contributing cause. Was this failure captured in a timely corrective action, or did it degrade undetected?
4. **Review recurrence pattern for air in-leakage** — two prior events on the same unit suggests the underlying boundary degradation mechanism is recurring. Consider enhanced surveillance frequency.

---
*Generated by the DACKAR RCA system — TSKR module demo. For analyst review only.*